# 1.DataSet Processing

In [1]:
import json
import os
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def load_from_json(filepath):
    """ Load a dictionary from a JSON file. """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def save_to_json(filepath, data):
    """ Save a dictionary to a JSON file. """
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def preprocess_text(text):
    """Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
    tokens = word_tokenize(text.lower())

    #TODO: numbers in claims and evidence may be useful, use isanum() instead of isalpha()?
    # filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalpha() and word not in stop_words]
    filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
    return " ".join(filtered_tokens)

In [3]:
EVIDENCE_FILE = 'data/evidence.json'
TRAIN_FILE = 'data/train-claims.json'

evidence_data = load_from_json(EVIDENCE_FILE)
claims_data = load_from_json(TRAIN_FILE)

In [6]:
# # Map evidence IDs to their texts
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}

# # Check whether directory already exists
# path = 'data/curated'
# if not os.path.exists(path):
#   os.mkdir(path)

# # Save preprocessed evidence file
# save_to_json(path + '/processed_evidence_map.json', evidence_map)


In [4]:
evidence_map = load_from_json('data/curated/processed_evidence_map.json')
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profession career age 16 eventu...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob 20 1936 profess...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd 6110 volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [5]:
data_for_dataframe = []
for claim_id, claim_details in claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
claims_df = pd.DataFrame(data_for_dataframe)
claims_df 

,claim,evidence
0,scientif evid co2 pollut higher co2 concentr a...,"[evidence-442946, evidence-1194317, evidence-1..."
1,el niño drove record high global temperatur su...,"[evidence-338219, evidence-1127398]"
2,1946 pdo switch cool phase,"[evidence-530063, evidence-984887]"
3,weather channel john coleman provid evid convi...,"[evidence-1177431, evidence-782448, evidence-5..."
4,januari 2008 cap 12 month period global temper...,"[evidence-1010750, evidence-91661, evidence-72..."
...,...,...
1223,climat scientist say aspect case hurrican harv...,"[evidence-1055682, evidence-1047356, evidence-..."
1224,5th assess report 2013 ipcc estim human emiss ...,[evidence-916755]
1225,sinc mid 1970 global temperatur warm around de...,"[evidence-403673, evidence-889933, evidence-11..."
1226,abnorm temperatur spike februari earlier month...,"[evidence-97375, evidence-562427, evidence-521..."


In [9]:
import random

# Vectorization
vectorizer = TfidfVectorizer()
all_texts = claims_df['claim'].tolist() + evidence_df['evidence'].tolist()

# sample_texts = random.sample(all_texts, 500)

# # Fit the vectorizer on both claims and evidences
# vectorizer.fit(sample_texts) 
# claim_vec = vectorizer.transform(claims_df['claim']).toarray() 
# claims_df['claim_tfidf'] = list(claim_vec) 
# claim_vec.shape

In [10]:

# evidence_vec = vectorizer.transform(evidence_df['evidence']).toarray()
# evidence_vec.shape

In [6]:
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# # We need data for training the model
# processed_sentences = [sent.split() for sent in all_texts]

# model = Word2Vec(
#     sentences=processed_sentences,
# )


tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(all_texts)]
tagged_data[:1]

NameError: name 'all_texts' is not defined

In [12]:
model = Doc2Vec(vector_size=50, min_count=1, epochs=20)
  
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
model.save("d2v.model")

In [13]:
model= Doc2Vec.load("d2v.model")

In [14]:
claims_df["vector"] = ""
for i in range(claims_df.shape[0]):
    inferred_vector = model.infer_vector(claims_df["claim"][i].split())
    claims_df["vector"][i] = inferred_vector
claims_df.to_csv('train_claim_vector.csv', index=False)
claims_df 

,claim,evidence,vector
0,scientif evid co2 pollut higher co2 concentr a...,"[evidence-442946, evidence-1194317, evidence-1...","[0.0146728605, 0.12625784, -0.11276962, -0.072..."
1,el niño drove record high global temperatur su...,"[evidence-338219, evidence-1127398]","[0.09453631, -0.11059547, -0.14010006, -0.3389..."
2,1946 pdo switch cool phase,"[evidence-530063, evidence-984887]","[0.02883816, 0.06266779, -0.062135715, -0.0004..."
3,weather channel john coleman provid evid convi...,"[evidence-1177431, evidence-782448, evidence-5...","[0.07255763, -0.06587273, -0.10453821, 0.21353..."
4,januari 2008 cap 12 month period global temper...,"[evidence-1010750, evidence-91661, evidence-72...","[0.14178367, 0.014683072, 0.07192291, -0.22664..."
...,...,...,...
1223,climat scientist say aspect case hurrican harv...,"[evidence-1055682, evidence-1047356, evidence-...","[0.087618195, 0.0039493246, -0.283926, -0.1802..."
1224,5th assess report 2013 ipcc estim human emiss ...,[evidence-916755],"[0.05541971, 0.0630518, -0.014345324, -0.10453..."
1225,sinc mid 1970 global temperatur warm around de...,"[evidence-403673, evidence-889933, evidence-11...","[0.060415942, -0.00991549, -0.31236076, -0.121..."
1226,abnorm temperatur spike februari earlier month...,"[evidence-97375, evidence-562427, evidence-521...","[0.028433701, 0.06975459, -0.24209118, -0.2778..."


In [15]:
evidence_df["vector"] = ""
for i in range(evidence_df.shape[0]):
    inferred_vector = model.infer_vector(evidence_df["evidence"][i].split())
    evidence_df["vector"][i] = inferred_vector
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[-0.10274349, -0.1521901, -0.14389612, -0.0647..."
1,evidence-1,lindberg began profession career age 16 eventu...,"[0.021682115, 0.09412701, -0.4038611, 0.014506..."
2,evidence-2,boston ladi cambridg vampir weekend,"[0.02425811, -0.04849113, 0.2777055, 0.0504023..."
3,evidence-3,gerald franci goyer born octob 20 1936 profess...,"[0.06251363, -0.26637015, -0.27424705, -0.1641..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.18131626, -0.11180243, -0.36687618, 0.10807..."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.044391014, -0.061329443, -0.169598, -0.0482..."
1208823,evidence-1208823,class fn org fyrd 6110 volda,"[0.039497066, -0.12655985, -0.3452346, -0.0604..."
1208824,evidence-1208824,dragon storm game game collect card game,"[0.00019951402, -0.16171727, -0.5241022, -0.10..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.17121354, -0.10472176, -0.53138727, 0.01106..."


In [16]:
evidence_df.to_csv('evidence_vector.csv', index=False) 

In [17]:
inferred_vector = model.infer_vector(tagged_data[0].words)
model.dv.most_similar([inferred_vector], topn=len(model.dv))

[('0', 0.8491038680076599),
 ('486870', 0.6858229637145996),
 ('505508', 0.6741513609886169),
 ('456801', 0.6693487763404846),
 ('253701', 0.6561805009841919),
 ('676209', 0.6514226198196411),
 ('791783', 0.6494963765144348),
 ('1022742', 0.6482139229774475),
 ('669420', 0.6476085186004639),
 ('782485', 0.6354619860649109),
 ('487135', 0.633489191532135),
 ('772661', 0.6323564052581787),
 ('110256', 0.6323369145393372),
 ('237548', 0.6318098902702332),
 ('836650', 0.6294390559196472),
 ('173293', 0.626121461391449),
 ('962769', 0.6254141926765442),
 ('1164705', 0.6250473260879517),
 ('936017', 0.6227185130119324),
 ('368255', 0.6211792230606079),
 ('532083', 0.6210688948631287),
 ('492540', 0.6207088232040405),
 ('16838', 0.6194145083427429),
 ('199208', 0.6190609931945801),
 ('890627', 0.6188330054283142),
 ('956128', 0.6175991892814636),
 ('383086', 0.6174643635749817),
 ('675489', 0.6173979640007019),
 ('914341', 0.6150795817375183),
 ('1166851', 0.6134316325187683),
 ('202507', 0.6

In [18]:
from gensim.models import KeyedVectors
model.save("word2vec.model")
word_vectors = model.wv
word_vectors.save("word2vec.wordvectors")

# Load back with memory-mapping = read-only, shared across processes.
wv = KeyedVectors.load("word2vec.wordvectors", mmap='r')

In [19]:
DEV_FILE = 'data/dev-claims.json'
dev_claims_data = load_from_json(DEV_FILE)

data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim,evidence
0,south australia expens electr world,"[evidence-67732, evidence-572512]"
1,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2..."
2,mean world 1c warmer time,"[evidence-889933, evidence-694262]"
3,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28..."
4,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...
149,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85..."
150,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20..."
151,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1..."
152,recent studi led lawrenc livermor nation labor...,[evidence-660755]


In [20]:
dev_claims_df["vector"] = ""
for i in range(dev_claims_df.shape[0]):
    inferred_vector = model.infer_vector(dev_claims_df["claim"][i].split())
    dev_claims_df["vector"][i] = inferred_vector

dev_claims_df 

,claim,evidence,vector
0,south australia expens electr world,"[evidence-67732, evidence-572512]","[-0.14695428, -0.09042706, -0.07665651, -0.079..."
1,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2...","[-0.112900116, -0.06112025, -0.2725012, -0.117..."
2,mean world 1c warmer time,"[evidence-889933, evidence-694262]","[0.03736557, -0.19243436, -0.17334826, -0.0584..."
3,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28...","[-0.09359704, 0.12776884, -0.0056210617, 0.047..."
4,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947...","[0.0400215, -0.029097028, -0.2028753, -0.06434..."
...,...,...,...
149,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85...","[0.28616574, 0.08560463, -0.035217457, -0.0863..."
150,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20...","[0.01725103, 0.021944348, -0.07149153, 0.07700..."
151,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1...","[-0.080627546, 0.10826158, -0.06843386, -0.110..."
152,recent studi led lawrenc livermor nation labor...,[evidence-660755],"[0.29741588, 0.2778654, -0.24676636, -0.173328..."


In [21]:
dev_claims_df['vector'].to_numpy()[0]

array([-0.14695428, -0.09042706, -0.07665651, -0.07978963,  0.12622364,
        0.00501365, -0.09204441, -0.01515245,  0.0946954 ,  0.09834569,
       -0.02270767, -0.07921322,  0.16500655, -0.01001099,  0.05098858,
       -0.12242872,  0.08092836, -0.06791194,  0.05861909, -0.11603323,
       -0.03153627,  0.09908666,  0.06579974,  0.03993439, -0.05210162,
        0.00666871, -0.22953184,  0.01598689,  0.03634018, -0.07705376,
       -0.14800063,  0.12090281, -0.13021603, -0.08185715,  0.1519484 ,
        0.08814736,  0.027516  ,  0.028056  ,  0.06988152,  0.00517484,
       -0.03458975,  0.00844377,  0.02469297, -0.00030825, -0.07978516,
       -0.0796921 ,  0.03141923, -0.06258724, -0.02241901,  0.06023757],
      dtype=float32)

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
X = np.array(dev_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
sim
    

array([[ 0.00117741,  0.24191558, -0.03670118, ...,  0.2770185 ,
         0.19545878, -0.09969829],
       [-0.19994286,  0.79901534, -0.48930073, ...,  0.870588  ,
         0.6441169 ,  0.07372354],
       [ 0.15423973,  0.07285927,  0.04803744, ...,  0.16585894,
         0.12644769,  0.05646985],
       ...,
       [-0.09713285,  0.15564387, -0.00675905, ...,  0.04100898,
         0.10649468,  0.00219995],
       [-0.00975218,  0.09400634, -0.2910451 , ...,  0.23731726,
         0.11471446,  0.17199133],
       [-0.00845153,  0.097124  , -0.12998559, ...,  0.2928345 ,
         0.16397814,  0.3564576 ]], dtype=float32)

In [23]:
for i in range(sim.shape[0]):
    print(np.where(sim[i]>0.7))

(array([  47847,   69090,   80646,   81037,   85519,   98593,  108925,
        144201,  148161,  163468,  163656,  164501,  169795,  179706,
        181500,  192479,  220223,  230320,  250017,  260583,  275430,
        279864,  282633,  293514,  349454,  358578,  374505,  382811,
        409221,  434774,  443311,  456796,  460668,  480899,  497068,
        501849,  558547,  572512,  585338,  597849,  599907,  600127,
        605361,  614499,  632706,  641060,  643464,  649294,  649584,
        679350,  684667,  697698,  713567,  716061,  718936,  744921,
        756198,  757863,  760144,  763705,  769999,  811839,  814661,
        819261,  825238,  844889,  857093,  866834,  872151,  910426,
        917387,  926163,  935211,  975241,  996254, 1023710, 1029971,
       1036628, 1044999, 1096974, 1131651, 1160482, 1162000, 1177471,
       1181118, 1182886]),)
(array([      1,       9,      21, ..., 1208816, 1208818, 1208824]),)
(array([], dtype=int64),)
(array([130492]),)
(array([  31748,